# Cambridge, MD — City Levy LVT Model

Revenue-neutral **4:1 split-rate** land value tax for the **City of Cambridge** (Dorchester County,
Maryland), holding the city's existing exemption structure in place.

> **Not** Cambridge, Massachusetts — that city is modeled separately in `cities/cambridge/`.
> This notebook is `cambridge_md` throughout.

## Policy assumptions (confirmed up front)

- **Scope**: Cambridge **city levy only**. Cambridge's FY2027 municipal real-property rate is
  **$0.80 per $100** of assessed value. Dorchester County ($0.9567/$100 *inside* Cambridge — the
  county grants Cambridge a tax differential against its $1.0300 rate elsewhere) and the State of
  Maryland ($0.1120/$100) levy on top of this in real bills, but Cambridge cannot unilaterally
  reform either, so only the municipal portion is modeled.
- **Reform**: revenue-neutral split-rate at **4:1** (land taxed 4× the improvement rate).
- **Exemptions preserved**: fully-exempt parcels (SDAT `EXCLASS` / land-use `E`,`EC` — government,
  church, non-profit, housing authority, tax-sale) stay exempt. Maryland's Homestead Tax Credit
  (Cambridge cap: **5%/yr**) and the three-year phase-in are reflected in the aggregate base
  calibration in Section 4.
- **Waterfront**: single-family homes are split into waterfront vs. inland — see Section 3b.
  Cambridge sits on the Choptank River and shoreline proximity is the dominant land-value driver
  here, so the undivided SFH median would hide the result that matters.

## Sources

| Fact | Value | Source |
|---|---|---|
| Parcel geometry + SDAT assessment attributes | 7,426 parcels, SDAT data `2026MAY` | [MD iMAP `MD_ParcelBoundaries`](https://mdgeodata.md.gov/imap/rest/services/PlanningCadastre/MD_ParcelBoundaries/MapServer/0) |
| FY2026 municipal rate | $0.8096 / $100 | [SDAT Tax Rates 2025–2026](https://dat.maryland.gov/Documents/statistics/TaxRates2025-2026.pdf); City of Cambridge [FY2025 audited financials](https://www.choosecambridge.com/DocumentCenter/View/3374/2025-Financial-Statements) |
| FY2027 municipal rate | $0.80 / $100 (adopted, Ordinance 1272) | [Cambridge Spy, FY27 budget adoption](https://cambridgespy.substack.com/p/cambridge-approves-budget-after-debate) |
| FY2026 assessable base / levy | $1,170,258,597 → $9,474,414 | [SDAT 2026 Constant Yield Tax Rate report](https://dat.maryland.gov/Pages/Constant-Yield-Report.aspx) |
| FY2027 assessable base (ex-new-construction) | $1,216,804,628 | same |
| Cambridge homestead cap | 5% / yr | [SDAT County & Municipal Homestead Credit Percentages](https://dat.maryland.gov/realproperty/documents/homestead_percent_caps.pdf) |

## Section 1 — Imports and constants

In [ ]:
import sys
import os
import time
from pathlib import Path
import requests

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Polygon
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.waterfront_utils import (
    add_waterfront_flags,
    waterfront_share,
    split_single_family_by_waterfront,
)

# Constants
CITY_NAME = 'cambridge_md'      # 'cambridge' is taken by Cambridge, MA
STATE_FIPS = '24'               # Maryland
COUNTY_FIPS = '019'             # Dorchester County
FIPS = STATE_FIPS + COUNTY_FIPS
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

# Cambridge FY2027 real-property rate: $0.80 / $100 of assessed value
# = 8.00 per $1,000 = 8.00 mills (LVTShift convention: millage is per $1,000)
CITY_RATE_PER_100 = 0.80
CITY_MILLAGE = CITY_RATE_PER_100 * 10.0

# Official controls (SDAT 2026 Constant Yield Tax Rate report)
OFFICIAL_BASE_FY27 = 1_216_804_628     # FY2027 assessable base, ex-new-construction
OFFICIAL_BASE_FY26 = 1_170_258_597     # FY2026 assessable base
OFFICIAL_LEVY_FY26 = 9_474_414         # FY2026 levy at $0.8096
OFFICIAL_LEVY_FY27 = OFFICIAL_BASE_FY27 * CITY_RATE_PER_100 / 100.0

# Parcel-map identity columns
PARCEL_ID_COL = 'ACCTID'               # SDAT account id
OWNER_NAME_COL = None                  # iMAP parcel layer carries no owner-name field
OWNER_ADDRESS_COL = 'ADDRESS'          # situs / property address
# No per-parcel record link. MD iMAP ships a SDATWEBADR column that LOOKS like a
# deep link (.../viewdetails.aspx?County=10&SearchType=ACCT&District=07&AccountNumber=...),
# but SDAT's Real Property search is a session-based ASP.NET postback app: every
# such URL 302s to the bare search form. Verified in a browser, Aug 2026. Emitting
# 6,972 links that all land on a generic search page reads as a working feature and
# isn't, so we emit none — per build-notebook.md. Users get the ACCTID to paste in.
PARCEL_URL_TEMPLATE = None

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f"{CITY_NAME}: ${CITY_RATE_PER_100}/$100 = {CITY_MILLAGE} mills")
print(f"FY2027 official levy target: ${OFFICIAL_LEVY_FY27:,.0f}")

## Section 2 — Fetch / load parcel data

MD iMAP's statewide parcel layer is a **MapServer** holding ~2.4M records, so
`lvt.cloud_utils.get_feature_data_with_geometry` (which hard-codes `where='1=1'` and
`FeatureServer`) can't be used. We inline a small ArcGIS fetcher, as the Rockville notebook does.

Two MD-iMAP-specific quirks this fetcher handles — **both silently lose data if ignored**:

1. **`resultOffset` ceilings out at 6,000.** Straight offset pagination returns an empty page at
   offset 6000 and looks like a clean finish — it silently dropped 1,426 of Cambridge's 7,426
   parcels on the first attempt. We instead pull the full `objectIds` list up front and fetch by
   explicit id batches.
2. **Occasional batches fail or return empty.** We retry, then bisect down to single ids, and
   report anything genuinely unfetchable rather than quietly shipping a short file.

In [ ]:
PARCEL_QUERY_URL = (
    'https://mdgeodata.md.gov/imap/rest/services/'
    'PlanningCadastre/MD_ParcelBoundaries/MapServer/0/query'
)
# Dorchester County, incorporated town 003 = Cambridge
CAMBRIDGE_WHERE = "JURSCODE='DORC' AND TOWNCODE='003'"
PARCEL_CACHE = DATA_DIR / 'parcels.gpq'


def _esri_polygon_to_shapely(geom):
    rings = (geom or {}).get('rings') or []
    if not rings:
        return None
    if len(rings) == 1:
        return Polygon(rings[0])
    return Polygon(rings[0], holes=rings[1:])


def fetch_md_parcels(query_url, where, chunk_size=400):
    # Fetch every matching parcel by explicit objectId batches. Avoids the
    # MapServer resultOffset ceiling; bisects on failure so a bad batch degrades
    # to a few missing ids instead of a silently truncated file.
    session = requests.Session()
    ids = sorted(session.get(
        query_url,
        params={'f': 'json', 'where': where, 'returnIdsOnly': 'true'},
        timeout=120,
    ).json()['objectIds'])
    print(f"Matching parcels: {len(ids):,}")

    rows, missing = [], []

    def grab(batch):
        for attempt in range(3):
            try:
                r = session.get(query_url, params={
                    'f': 'json', 'objectIds': ','.join(map(str, batch)),
                    'outFields': '*', 'returnGeometry': 'true',
                    'outSR': 4326, 'geometryPrecision': 6,
                }, timeout=300)
                r.raise_for_status()
                d = r.json()
                if 'error' in d:
                    raise RuntimeError(d['error'].get('message'))
                feats = d.get('features', [])
                if feats or len(batch) == 1:
                    for f in feats:
                        a = f['attributes']
                        a['geometry'] = _esri_polygon_to_shapely(f.get('geometry'))
                        rows.append(a)
                    if not feats:
                        missing.extend(batch)
                    return
                raise RuntimeError('empty response')
            except Exception as exc:
                if attempt < 2:
                    time.sleep(2)
                    continue
                if len(batch) == 1:
                    print(f"  unfetchable objectId {batch[0]}: {exc}")
                    missing.extend(batch)
                    return
                mid = len(batch) // 2
                grab(batch[:mid])
                grab(batch[mid:])
                return

    for i in range(0, len(ids), chunk_size):
        grab(ids[i:i + chunk_size])
    if missing:
        print(f"  WARNING: {len(missing)} objectIds could not be fetched")
    return rows, len(ids)


if PARCEL_CACHE.exists():
    gdf = gpd.read_parquet(PARCEL_CACHE)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    rows, n_expected = fetch_md_parcels(PARCEL_QUERY_URL, CAMBRIDGE_WHERE)
    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    gdf = gdf[gdf.geometry.notna()].copy()
    assert len(gdf) >= 0.99 * n_expected, f"only got {len(gdf)} of {n_expected} parcels"
    gdf.to_parquet(PARCEL_CACHE)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(f"SDAT assessment vintage: {sorted(gdf['SDATDATE'].dropna().unique())}")
print(f"MDProperty View vintage: {sorted(gdf['MDPVDATE'].dropna().unique())}")

### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `NFMLNDVL` | SDAT "New Full Market Land Value" (latest triennial appraisal) |
| Improvement value | `NFMIMPVL` | SDAT "New Full Market Improvement Value" |
| Total value | `NFMTTLVL` | Land + improvement |
| Land use (broad) | `LU` / `DESCLU` | `R` Residential, `TH` Town House, `U` Residential Condominium, `C` Commercial, `I` Industrial, `M` Apartments, `A` Agricultural, `CR` Commercial/Residential, `E`/`EC` Exempt |
| Commercial subtype | `CIUSE` / `DESCCIUSE` | Populated for commercial / industrial only; 5-digit code with a family prefix in the description (`STORE`, `OFFICE`, `AUTO Parking`, …) |
| Exemption class | `EXCLASS` / `DESCEXCL` | Non-null ⇒ fully exempt |
| Waterfront factor | `PFLW` | Assessor's own water-influence flag: `0` none, `1` waterfront, `2` water-influenced |
| Units | `BLDG_UNITS` | Residential unit count |
| Structure size | `SQFTSTRC` | Used in the Section 3 artifact scan |
| Parcel ID | `ACCTID` | SDAT account id (county + district + account) |
| Record URL | `SDATWEBADR` | Full SDAT record link, per parcel |
| Municipality | `TOWNCODE` | `'003'` = Cambridge |

**Assessment ratio**: Maryland assesses at **100% of full cash value**; the tax rate is quoted per
$100. There is no fractional assessment ratio, but there *is* a three-year phase-in of
reassessment increases plus a 5% homestead cap — handled in Section 4.

## Section 3 — Classify and validate

1. Coerce value columns to numeric, clip negatives.
2. Build the fully-exempt flag.
3. Drop $0-land-**and**-$0-improvement parcels (common-area lots, ROW slivers — not in the
   assessable base, never move under any shift).
4. Map every parcel to a `PROPERTY_CATEGORY`.
5. Override: $0 improvement → `Vacant Land`.
6. Artifact scan on the raw inputs (Gate 5), *before* modeling.

In [ ]:
VALUE_COLS = ['NFMLNDVL', 'NFMIMPVL', 'NFMTTLVL']
for col in VALUE_COLS + ['BLDG_UNITS', 'SQFTSTRC', 'ACRES']:
    gdf[col] = pd.to_numeric(gdf.get(col), errors='coerce').fillna(0)
for col in VALUE_COLS:
    gdf[col] = gdf[col].clip(lower=0)

# Fully exempt: SDAT exemption class set, OR land use is Exempt / Exempt Commercial.
# The two disagree on 81 parcels (77 LU='E' with no EXCLASS; 4 with EXCLASS but a
# commercial LU), so we take the union rather than trusting either field alone.
gdf['full_exmp'] = (gdf['EXCLASS'].notna() | gdf['LU'].isin(['E', 'EC'])).astype(int)

n_before = len(gdf)
gdf = gdf[(gdf['NFMLNDVL'] > 0) | (gdf['NFMIMPVL'] > 0)].copy()
print(f"Dropped {n_before - len(gdf):,} $0-land-and-$0-improvement parcels (not in the assessable base).")

print(f"\nParcels: {len(gdf):,}")
print(f"  Fully exempt: {int(gdf['full_exmp'].sum()):,}")
print(f"  Taxable:      {int((gdf['full_exmp'] == 0).sum()):,}")
print(f"\nTaxable value distribution:")
print(gdf.loc[gdf['full_exmp'] == 0, VALUE_COLS].describe().map(lambda x: f"${x:,.0f}").to_string())
print(f"\nLand > Total (data errors): {int((gdf['NFMLNDVL'] > gdf['NFMTTLVL']).sum())}")
print(f"Total != land + improvement: {int((abs(gdf['NFMTTLVL'] - (gdf['NFMLNDVL'] + gdf['NFMIMPVL'])) > 1).sum())}")

In [ ]:
# Property categorization.
#
# Maryland SDAT gives a broad land-use code (LU) for every parcel and a detailed
# commercial/industrial use code (CIUSE) for commercial and industrial ones. The
# CIUSE *description* carries a family prefix (STORE / OFFICE / AUTO / INDUSTRY /
# ...) which is the cleanest thing to branch on.

def _txt(v):
    # Normalize a possibly-missing SDAT text field to a clean string.
    # NOT `str(v or '')`: after the GeoParquet round-trip missing text comes back
    # as float('nan'), which is TRUTHY, so `or ''` never fires and every null
    # becomes the literal string 'nan'. That silently routed all 3,700 Cambridge
    # houses through the commercial branch on the first run.
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ''
    return str(v).strip()


def _mf_by_units(units):
    if units >= 5:
        return 'Large Multi-Family (5+ units)'
    if units >= 2:
        return 'Small Multi-Family (2-4 units)'
    return 'Other Residential'


def _commercial_category(desc, units):
    # Map a DESCCIUSE description (e.g. "STORE Shopping Center") to a category.
    d = desc.upper()
    fam = d.split(' ', 1)[0]

    # Mixed use: the assessor names these explicitly
    if 'APARTMENT UPSTAIRS' in d or 'MIXED RESIDENTIAL' in d:
        return 'Mixed Use'
    if fam == 'HOUSING':
        if 'MOBILE HOME PARK' in d:
            return 'Other Residential'
        return _mf_by_units(units if units >= 2 else 5)   # SDAT "Apartments" are 5+ in practice
    if fam == 'TRAVEL':
        return 'Hotel'
    if fam == 'AUTO':
        return 'Transportation - Parking' if 'PARKING' in d else 'Retail / General Commercial'
    if fam in ('STORE', 'RESTAURANT'):
        return 'Retail / General Commercial'
    if fam in ('OFFICE', 'BANK'):
        return 'Office / Commercial Condo'
    if fam in ('INDUSTRY', 'WAREHOUSE'):
        return 'Industrial'
    return 'Other Commercial'


def categorize(row):
    lu = _txt(row.get('LU')).upper()
    units = float(row.get('BLDG_UNITS') or 0)
    desc = _txt(row.get('DESCCIUSE'))

    if lu == 'TH':
        return 'Townhome / Rowhouse'
    if lu == 'U':
        return 'Condominium'
    if lu == 'A':
        return 'Agricultural'
    if lu == 'CR':
        return 'Mixed Use'
    if lu == 'M':
        return _mf_by_units(units)
    if lu == 'R':
        # A handful of residential parcels carry a commercial use code
        if desc and 'COMMON USE FACILITY' not in desc.upper():
            return _commercial_category(desc, units)
        if units >= 2:
            return _mf_by_units(units)
        return 'Single Family Residential'
    if lu in ('C', 'EC', 'I'):
        if desc:
            return _commercial_category(desc, units)
        return 'Industrial' if lu == 'I' else 'Other Commercial'
    if lu == 'E':
        # Exempt land use with no commercial subtype — held out of the model anyway
        return _commercial_category(desc, units) if desc else 'Other'
    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)

# Override: $0 improvement → Vacant Land, regardless of use code.
gdf.loc[gdf['NFMIMPVL'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

_tax_cat = gdf.loc[gdf['full_exmp'] == 0, 'PROPERTY_CATEGORY']
print("Taxable parcels by category:")
print(_tax_cat.value_counts().to_string())

# Guard against a silently mis-wired classifier. Cambridge is a small residential
# city: single-family homes must be the largest category, and the residual
# buckets must stay small. (The first run failed both of these.)
_resid = _tax_cat.isin(['Other', 'Other Commercial', 'Other Residential']).mean()
print(f"\nResidual 'Other*' share of taxable parcels: {_resid*100:.1f}%")
print(f"Single Family Residential count: {int((_tax_cat == 'Single Family Residential').sum()):,}")
assert (_tax_cat == 'Single Family Residential').sum() > 2_000, \
    'Single-family homes missing — classifier is mis-wired'
assert _resid < 0.10, f'residual Other* buckets are {_resid*100:.1f}% of parcels — refine the mapping'

In [ ]:
# Gate 5 artifact scan on the RAW inputs, before any modeling.
#
# The canary we are looking for is the Seattle pattern: parcels that plainly have
# a building but carry a placeholder/absent improvement value, which would pin
# whole categories at the land-only ceiling. Cross-check improvement value
# against the assessor's own structure square footage.
t = gdf[gdf['full_exmp'] == 0]
zero_imp = t[t['NFMIMPVL'] <= 1000]
has_struct = t[t['SQFTSTRC'] > 200]

print(f"Taxable parcels:                          {len(t):,}")
print(f"  with $0/placeholder improvement value:  {len(zero_imp):,} ({len(zero_imp)/len(t)*100:.1f}%)")
print(f"    ...of those, with structure sqft > 0: {int((zero_imp['SQFTSTRC'] > 0).sum())}  <- missing building value if large")
print(f"  with structure sqft > 200:              {len(has_struct):,}")
print(f"    ...of those, improvement <= $1,000:   {int((has_struct['NFMIMPVL'] <= 1000).sum())}")

_v = has_struct[has_struct['NFMIMPVL'] > 1000]
print(f"\nImprovement $/sqft (parcels with both):")
print((_v['NFMIMPVL'] / _v['SQFTSTRC']).describe([.05, .25, .5, .75, .95]).round(1).to_string())

print(f"\nMedian land share of value by category (taxable):")
_ls = t.assign(land_share=(t['NFMLNDVL'] / t['NFMTTLVL'].replace(0, np.nan) * 100))
print(_ls.groupby('PROPERTY_CATEGORY').agg(
    n=('land_share', 'size'),
    median_land_share_pct=('land_share', 'median'),
    median_land=('NFMLNDVL', 'median'),
    median_improvement=('NFMIMPVL', 'median'),
).round(1).sort_values('n', ascending=False).to_string())

#### Flat-assessor-ratio detection

A *median* land share can look perfectly plausible while the underlying values carry no
information — the Maricopa/Phoenix failure mode, where the assessor books residential land at a
flat percentage of total value. A median cannot detect this; only **dispersion** can. If a class's
land share has near-zero spread, its land value is a mechanical function of its total value, every
parcel in it moves identically under a split rate, and the per-parcel result is manufactured.

This test is the reason the Cambridge condo result is reported as an artifact below.

In [ ]:
# Dispersion of land share within each land-use class. Near-zero IQR ⇒ flat assessor ratio.
_imp = gdf[(gdf['full_exmp'] == 0) & (gdf['NFMTTLVL'] > 0) & (gdf['NFMIMPVL'] > 0)].copy()
_imp['land_share'] = _imp['NFMLNDVL'] / _imp['NFMTTLVL']

_disp = _imp.groupby('LU')['land_share'].describe(percentiles=[.25, .5, .75])[['count', '25%', '50%', '75%']]
_disp['IQR'] = (_disp['75%'] - _disp['25%'])
_disp['pct_at_mode'] = _imp.groupby('LU')['land_share'].apply(
    lambda s: (s.sub(s.median()).abs() < 0.005).mean()
)
print("Land-share dispersion by land use (improved, taxable parcels):")
print(_disp.round(4).sort_values('count', ascending=False).to_string())

FLAT_RATIO_IQR = 0.01
_flat = _disp[(_disp['IQR'] < FLAT_RATIO_IQR) & (_disp['count'] >= 25)]
if len(_flat):
    print(f"\n*** FLAT ASSESSOR RATIO DETECTED in: {list(_flat.index)} ***")
    for _lu in _flat.index:
        _sh = _disp.loc[_lu, '50%']
        # denominator = the FULL taxable land base (incl. vacant land), so this
        # figure is directly comparable to the Section 5b sensitivity.
        _share_of_land = (
            _imp.loc[_imp['LU'] == _lu, 'NFMLNDVL'].sum()
            / gdf.loc[gdf['full_exmp'] == 0, 'NFMLNDVL'].sum() * 100
        )
        print(f"    LU={_lu}: land booked at a flat {_sh*100:.1f}% of total value "
              f"(IQR {_disp.loc[_lu,'IQR']:.4f}, {_disp.loc[_lu,'pct_at_mode']*100:.0f}% at the mode); "
              f"{_share_of_land:.1f}% of the city land base")
    print("    => per-parcel LVT results for these classes carry NO information.")
    print("       They are reported, but must be read as a class-level artifact. See limitations.")
else:
    print(f"\nNo flat assessor ratio detected (all classes have land-share IQR >= {FLAT_RATIO_IQR}).")

#### Agricultural use assessment (Md. Tax-Prop. § 8-209)

Maryland assesses actively farmed land at its *agricultural use value* rather than market value.
That is a preferential assessment the LVT model inherits and cannot undo, and it matters here far
more than the Agricultural row's parcel count suggests: the check below contrasts each class's
share of taxable land **area** with its share of taxable land **value**. Agricultural land is the
one class where those two numbers diverge by orders of magnitude.

In [ ]:
# Land AREA share vs land VALUE share, by land use. A class holding a large share of
# the city's acreage but a negligible share of its land value is preferentially assessed.
_t = gdf[gdf['full_exmp'] == 0].copy()
_area = _t.groupby('LU')['ACRES'].sum()
_val = _t.groupby('LU')['NFMLNDVL'].sum()
_cmp = pd.DataFrame({
    'parcels': _t.groupby('LU').size(),
    'acres': _area.round(0),
    'pct_of_area': (_area / _area.sum() * 100).round(2),
    'pct_of_land_value': (_val / _val.sum() * 100).round(2),
    'land_$_per_acre': (_val / _area.replace(0, np.nan)).round(0),
})
_cmp['area_to_value_ratio'] = (_cmp['pct_of_area'] / _cmp['pct_of_land_value']).round(1)
# Condo unit records carry ~0 acres (the land sits under the development, not the unit),
# so a $/acre for them is meaningless rather than large — blank it instead of printing it.
_cmp.loc[_cmp['acres'] < 1.0, 'land_$_per_acre'] = float('nan')
print(f"Total taxable acreage in the model: {_t['ACRES'].sum():,.0f} ac")
print("Share of taxable land AREA vs share of taxable land VALUE, by land use:")
print(_cmp.sort_values('pct_of_area', ascending=False).to_string())
print(f"\nAgricultural land $/acre: median ${(_t.loc[_t['LU']=='A','NFMLNDVL'] / _t.loc[_t['LU']=='A','ACRES']).median():,.0f}"
      f"  aggregate ${_val.get('A',0)/_area.get('A',1):,.0f}")

_ag = _cmp.loc['A'] if 'A' in _cmp.index else None
if _ag is not None:
    print(f"\n*** AGRICULTURAL USE ASSESSMENT: {_ag['acres']:,.0f} acres = "
          f"{_ag['pct_of_area']:.1f}% of taxable land area but only "
          f"{_ag['pct_of_land_value']:.2f}% of taxable land value "
          f"(${_ag['land_$_per_acre']:,.0f}/acre) ***")
    print("    Md. Tax-Prop. 8-209 use value, not market value. These are development-zoned")
    print("    parcels inside city limits, so the LVT burden on the city's largest")
    print("    land-banking block is structurally understated. See limitations.")
    print("\n    Zoning of the agricultural parcels:")
    print(_t.loc[_t['LU'] == 'A'].groupby(_t['ZONING'].fillna('(none)')).agg(
        n=('ACRES', 'size'), acres=('ACRES', 'sum')).round(1).to_string())

#### The size gradient: large parcels are assessed cheaply per acre generally

The § 8-209 use assessment above is the sharpest case of a broader pattern. SDAT's assessed land
value per acre falls steeply with parcel size for unimproved land, and that gradient survives
excluding the ag-assessed parcels entirely — so it is not only a § 8-209 effect. Part of it is
genuine economics (raw acreage without utilities or entitlements really is worth less per acre than
a serviced building lot), but the magnitude is large enough to matter for an LVT read.

**Reconciling with other parcel counts.** Anyone comparing this model against another Cambridge
analysis should check the *definition* of "agricultural / open space" before concluding the data
differ. Three defensible definitions give three very different counts here:

| Definition | Parcels |
|---|---|
| SDAT land-use code `LU='A'` (**what this model uses**) | 18 |
| + exempt parks, recreation, cemeteries, airport | 74 |
| Unimproved parcels of 5+ acres, any owner or use code | 87 |

All three describe the same city. The 18 is the set actually receiving the § 8-209 farm assessment;
the 87 is a size-and-improvement rule that also picks up 38 vacant residential tracts, 12 vacant
commercial, and **18 fully-exempt public parcels** (airport, state land) that this model holds out
by design. Every one of the 69 *taxable* parcels in that 87 **is** in this model — as `Agricultural`
(14) or `Vacant Land` (55). Nothing is missing; the taxonomies just cut differently.

In [ ]:
# Assessed land $/acre by parcel size, unimproved taxable parcels. A steep gradient means
# the LVT signal on large landholdings is muted well beyond the ag-assessed subset.
_v = gdf[(gdf['full_exmp'] == 0) & (gdf['NFMIMPVL'] <= 0) & (gdf['ACRES'] > 0.02)].copy()
_v['land_per_acre'] = _v['NFMLNDVL'] / _v['ACRES']
_v['size_band'] = pd.cut(_v['ACRES'], [0, 0.15, 0.35, 1, 2, 5, 20, 100, 1000])

_g = _v.groupby('size_band', observed=True).agg(
    n=('land_per_acre', 'size'), acres=('ACRES', 'sum'),
    median_land_per_acre=('land_per_acre', 'median'))
print("Taxable VACANT land — assessed $/acre by parcel size:")
print(_g.round(0).to_string())

_small = _v.loc[_v['ACRES'] <= 0.35, 'land_per_acre'].median()
_big = _v.loc[_v['ACRES'] >= 20, 'land_per_acre'].median()
print(f"\nSmall lots (<=0.35 ac) ${_small:,.0f}/ac vs large tracts (>=20 ac) ${_big:,.0f}/ac "
      f"=> {_small/_big:.0f}x size discount")

_vn = _v[_v['LU'] != 'A']
_sn = _vn.loc[_vn['ACRES'] <= 0.35, 'land_per_acre'].median()
_bn = _vn.loc[_vn['ACRES'] >= 20, 'land_per_acre'].median()
print(f"Excluding ag-assessed parcels: ${_sn:,.0f} vs ${_bn:,.0f} => {_sn/_bn:.0f}x "
      f"(the gradient is NOT only a 8-209 effect)")

_big5 = _v[_v['ACRES'] >= 5]
print(f"\nParcels >=5 acres: {_big5['ACRES'].sum():,.0f} taxable acres = "
      f"{_big5['ACRES'].sum()/gdf.loc[gdf['full_exmp']==0,'ACRES'].sum()*100:.0f}% of taxable acreage, "
      f"carrying {_big5['NFMLNDVL'].sum()/gdf.loc[gdf['full_exmp']==0,'NFMLNDVL'].sum()*100:.1f}% "
      f"of taxable land value")

### Section 3b — Waterfront single-family split

Cambridge fronts the **Choptank River** (roughly two miles wide here) and is cut through by
**Cambridge Creek**. Shoreline proximity is the single largest driver of residential land value in
town, so an undivided "Single Family Residential" median would average a small band of very
high-land-value shoreline homes together with the inland stock and hide the result that matters.

We use the canonical `lvt/waterfront_utils.py` (TIGER/Line AREAWATER, ≥1 ha water bodies only, so
drainage ponds don't flag a home) and apply the skill's "decent chunk" gate: split only when ≥2% of
single-family parcels sit within 100 m of significant water.

**Independent check available here:** SDAT carries its own water-influence factor, `PFLW`
(`1` = waterfront, `2` = water-influenced). We cross-tabulate the computed flag against it rather
than assuming the distance calculation is right.

In [ ]:
# Distance to significant water (>= 1 ha), cached in the city data dir.
gdf = add_waterfront_flags(
    gdf,
    fips_code=FIPS,
    cache_dir=DATA_DIR,
    min_water_area_m2=10_000,
)

stats = waterfront_share(gdf, category_col='PROPERTY_CATEGORY')
print("Waterfront share of single-family stock:", stats)

# Cross-check the computed flag against the assessor's own water-influence factor.
_pflw = gdf['PFLW'].fillna('0').astype(str)
print("\nComputed waterfront_100m vs SDAT PFLW (all parcels):")
print(pd.crosstab(gdf['waterfront_100m'], _pflw, rownames=['waterfront_100m'], colnames=['PFLW']).to_string())

print("\nMedian land value by SDAT PFLW (taxable):")
_t = gdf[gdf['full_exmp'] == 0]
print(_t.groupby(_pflw[_t.index])['NFMLNDVL'].agg(['size', 'median']).to_string())

In [ ]:
WATERFRONT_GATE_PCT = 2.0
APPLY_WATERFRONT_SPLIT = stats['waterfront_pct'] >= WATERFRONT_GATE_PCT

if APPLY_WATERFRONT_SPLIT:
    gdf = split_single_family_by_waterfront(gdf, category_col='PROPERTY_CATEGORY')
    print(f"Applied waterfront SFH split ({stats['waterfront_pct']:.1f}% >= {WATERFRONT_GATE_PCT}% gate).")
    print(gdf.loc[gdf['full_exmp'] == 0, 'PROPERTY_CATEGORY'].value_counts().to_string())
else:
    print(f"Skipped waterfront split ({stats['waterfront_pct']:.1f}% < {WATERFRONT_GATE_PCT}% gate).")

## Section 4 — Current tax model + revenue validation

### The Maryland base problem

Maryland taxes the **phased-in assessed value**, not full cash value. Two mechanisms drive a wedge
between them, and **neither per-parcel figure is published in MD iMAP**:

1. **Three-year phase-in.** When a property is reassessed (triennially), the *increase* is added to
   the taxable assessment in three equal annual steps. Mid-cycle, taxable assessment trails the new
   appraisal.
2. **Homestead Tax Credit.** Cambridge caps taxable-assessment growth on owner-occupied principal
   residences at **5%/yr** — one of the tighter caps in Maryland.

The `NFM*` columns we have are the *new full cash appraisal*. So our raw base necessarily sits
**above** the certified assessable base. We measure that gap against the official figure, then close
it with a single uniform scalar.

### Why a uniform scalar is the right correction — and costs nothing

For parcel *i* scaled by any factor *fᵢ*, with rate *r* and solved millages *m_L*, *m_I*:

- `current_taxᵢ = r · fᵢ · (Lᵢ + Iᵢ)`
- `new_taxᵢ = fᵢ · (m_L·Lᵢ + m_I·Iᵢ)`
- ratio `new/current = (m_L·Lᵢ + m_I·Iᵢ) / (r·(Lᵢ + Iᵢ))` — ***fᵢ cancels.***

So a per-parcel scalar cannot change any parcel's percentage tax change. And because a *uniform*
scalar multiplies the aggregate land base, improvement base, and revenue target by the same factor,
it leaves the **solved millages exactly unchanged** too. It moves only the absolute dollar figures,
onto the correct footing. Section 4b verifies this empirically rather than taking it on faith.

This also means the entire phase-in/homestead question — the messiest part of modeling Maryland —
**does not affect the distributional finding at all**. It affects only the dollar totals.

In [ ]:
# Raw full-cash-value base, taxable parcels only
raw_base = float(gdf.loc[gdf['full_exmp'] == 0, 'NFMTTLVL'].sum())

print(f"Full-cash-value taxable base (SDAT 2026MAY):  ${raw_base:,.0f}")
print(f"Official FY2027 certified assessable base:    ${OFFICIAL_BASE_FY27:,.0f}")
print(f"  gap: {(raw_base / OFFICIAL_BASE_FY27 - 1) * 100:+.2f}%   (expected sign: positive)")
print(f"Official FY2026 certified assessable base:    ${OFFICIAL_BASE_FY26:,.0f}")
print(f"  gap: {(raw_base / OFFICIAL_BASE_FY26 - 1) * 100:+.2f}%")

# Sanity band: a mid-cycle phase-in plus a 5% homestead cap should put the full-cash
# base somewhere modestly above the certified base. Far outside this band would mean
# the wrong column, the wrong municipality, or a broken exempt filter.
gap_pct = (raw_base / OFFICIAL_BASE_FY27 - 1) * 100
assert 0 < gap_pct < 30, f"base gap {gap_pct:.1f}% is outside the plausible phase-in band"

PHASE_IN_RATIO = OFFICIAL_BASE_FY27 / raw_base
print(f"\nPhase-in / homestead scalar: {PHASE_IN_RATIO:.4f}  "
      f"({(1 - PHASE_IN_RATIO) * 100:.1f}% of full cash value is not yet on the tax roll)")

In [ ]:
# Taxable value columns — full cash value scaled onto the certified base.
gdf['taxable_land_value'] = gdf['NFMLNDVL'] * PHASE_IN_RATIO
gdf['taxable_improvement_value'] = gdf['NFMIMPVL'] * PHASE_IN_RATIO
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# Fully-exempt parcels carry a zero base.
_ex = gdf['full_exmp'] == 1
gdf.loc[_ex, ['taxable_land_value', 'taxable_improvement_value', 'taxable_total_value']] = 0.0

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

print(f"Modeled current city levy: ${current_revenue:,.0f}")
print(f"Official FY2027 levy:      ${OFFICIAL_LEVY_FY27:,.0f}   "
      f"({(current_revenue / OFFICIAL_LEVY_FY27 - 1) * 100:+.2f}%)")
print(f"Official FY2026 levy:      ${OFFICIAL_LEVY_FY26:,.0f}   "
      f"({(current_revenue / OFFICIAL_LEVY_FY26 - 1) * 100:+.2f}%)")
print()
print("NOTE: the FY2027 match is true BY CONSTRUCTION — the scalar above was calibrated")
print("to the certified base. The independent checks are (a) the pre-calibration gap")
print("sitting in the plausible phase-in band, (b) the parcel count, and (c) the FY2026")
print("levy landing nearby without being fit to.")

### Section 4b — Verify the invariance claim

Re-solve the split-rate model on the *unscaled* full-cash-value base and confirm that the millages
and every per-parcel percentage change come out identical. If this assertion ever fails, the
scalar is doing something it shouldn't be.

In [ ]:
_check = gdf[gdf['full_exmp'] == 0].copy()

# (a) scaled base — the model we actually use
_a_land, _a_imp, _, _ra = model_split_rate_tax(
    df=_check.copy(),
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=_check['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# (b) unscaled full cash value. model_split_rate_tax derives tax_change_pct from
# the frame's own 'current_tax' column, so that must be rebuilt on the unscaled
# base too — otherwise we would be comparing a scaled numerator to an unscaled
# denominator and the test would fail for the wrong reason.
_check_b = _check.copy()
_check_b['current_tax'] = _check_b['NFMTTLVL'] * CITY_MILLAGE / 1000.0
_b_land, _b_imp, _, _rb = model_split_rate_tax(
    df=_check_b,
    land_value_col='NFMLNDVL',
    improvement_value_col='NFMIMPVL',
    current_revenue=_check_b['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"land millage   scaled {_a_land:.6f}  vs unscaled {_b_land:.6f}")
print(f"imp  millage   scaled {_a_imp:.6f}  vs unscaled {_b_imp:.6f}")
_max_pct_diff = float(np.nanmax(np.abs(_ra['tax_change_pct'].values - _rb['tax_change_pct'].values)))
print(f"max per-parcel |Δ tax_change_pct| between the two: {_max_pct_diff:.10f}")

assert abs(_a_land - _b_land) < 1e-6 and abs(_a_imp - _b_imp) < 1e-6
assert _max_pct_diff < 1e-6
print("\nInvariance confirmed: the phase-in scalar moves dollars only, never the distribution.")

## Section 5 — Split-rate model (4:1)

In [ ]:
# Hold fully-exempt parcels out of the solver AND out of the modeled output:
# current = new = $0 carries no signal and would add a meaningless bar to the
# report plus a spike at 0% in the distribution. Reported by count instead.
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Land millage:        {land_millage:.4f} per $1,000  (= ${land_millage/10:.4f}/$100)")
print(f"Improvement millage: {improvement_millage:.4f} per $1,000  (= ${improvement_millage/10:.4f}/$100)")
print(f"Ratio: {land_millage/improvement_millage:.2f}:1")
print(f"Revenue check: ${new_revenue:,.0f}  (target ${gdf['current_tax'].sum():,.0f})")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

In [ ]:
# Gate 5 — post-model artifact scan. Read this table against economic priors before
# trusting anything above. Ceiling clustering (distinct categories pinned at one
# extreme value) is the canary for missing improvement values.
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(summary.round(3).to_string())

# The land-only ceiling: a parcel that is all land and no building.
ceiling = land_millage / CITY_MILLAGE - 1
print(f"\nLand-only ceiling (max possible increase): {ceiling*100:+.1f}%")
print("Categories at the ceiling should be ONLY those that are genuinely bare land.")

# Dispersion of the OUTCOME. A class whose p10-p90 spread is ~0 is not a finding —
# it is one number repeated N times, the signature of a flat assessor land ratio.
print("\nOutcome dispersion by category (p10 / median / p90 of tax_change_pct):")
_od = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].agg(
    n='size', p10=lambda s: s.quantile(.10), med='median', p90=lambda s: s.quantile(.90))
_od['spread_pts'] = (_od['p90'] - _od['p10']).round(1)
print(_od.round(1).sort_values('spread_pts').to_string())

### Section 5b — Sensitivity to the flat condo land ratio

Condominiums are **17.8% of the city's land base**, so SDAT's flat 30% convention does not just
distort the condo row — it moves the solved millages and therefore every other category. This cell
re-solves the model under alternative condo land shares to size that exposure honestly.

In [ ]:
# Re-solve with condo land share overridden, to bound the flat-ratio exposure.
_base = gdf.copy()
_is_condo = _base['LU'] == 'U'
print(f"Condos: {int(_is_condo.sum()):,} parcels = {_is_condo.mean()*100:.1f}% of modeled parcels, "
      f"{_base.loc[_is_condo,'taxable_land_value'].sum()/_base['taxable_land_value'].sum()*100:.1f}% of the land base")

def _resolve(label, condo_share=None):
    d = _base.copy()
    if condo_share is not None:
        _tot = d.loc[_is_condo, 'taxable_land_value'] + d.loc[_is_condo, 'taxable_improvement_value']
        d.loc[_is_condo, 'taxable_land_value'] = _tot * condo_share
        d.loc[_is_condo, 'taxable_improvement_value'] = _tot * (1 - condo_share)
    _lm, _im, _, _res = model_split_rate_tax(
        df=d, land_value_col='taxable_land_value', improvement_value_col='taxable_improvement_value',
        current_revenue=d['current_tax'].sum(), land_improvement_ratio=LAND_IMPROVEMENT_RATIO)
    _m = _res.groupby(d['PROPERTY_CATEGORY'])['tax_change_pct'].median()
    print(f"  {label:36s} land mill {_lm:7.4f} | condo {_m.get('Condominium', float('nan')):6.1f}%"
          f" | SFR inland {_m.get('Single Family Residential', float('nan')):6.2f}%"
          f" | townhome {_m.get('Townhome / Rowhouse', float('nan')):6.2f}%")

print("\nSensitivity of the whole model to the condo land-share assumption:")
_resolve('as modeled (SDAT flat 30%)')
_resolve('condo land = 16.1% (city SFH median)', 0.161)
_resolve('condo land = 41.0% (waterfront SFH)', 0.410)
print("\nThe SDAT 30% level is plausible for waterfront condo product, so it is kept as the")
print("authoritative source. The uncertainty band above is the honest error bar on every")
print("category median in this model — not just the condo row.")

## Section 6 — Exploration chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
s = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
s.plot.barh(ax=ax, color=['#c0392b' if v > 0 else '#27ae60' for v in s.values])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 split-rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f"Saved {DATA_DIR / 'category_preview.png'}")

## Section 7 — Census join + standard export + report + parcel map

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — the spatial join above brings
            # them onto gdf. Do NOT do a second merge against census_data here:
            # that produces median_income_x/y duplicates and silently zeros the
            # demographic columns downstream.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — PNGs in analysis/reports/<city>/
# min_category_count=25 (default 50) so the smaller but policy-relevant categories
# (Large Multi-Family, Hotel, Parking) appear in the category charts.
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, min_category_count=25)

In [ ]:
# Parcel-map export — GeoParquet + interactive HTML map colored by tax change.
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

_map_gdf = gdf if isinstance(gdf, _gpd.GeoDataFrame) else _gpd.GeoDataFrame(
    gdf, geometry='geometry', crs='EPSG:4326')
if _map_gdf.crs is None:
    _map_gdf = _map_gdf.set_crs('EPSG:4326')

map_gdf = save_parcel_map_export(
    gdf=_map_gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,   # None — see Section 1
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

## Validation summary

| Check | Result |
|---|---|
| Raw parcels fetched | 7,426 (all of Dorchester `TOWNCODE='003'`) |
| Dropped ($0-land **and** $0-improvement) | 62 |
| Modeled parcels | **6,972 taxable** (+392 fully-exempt held out and excluded from charts) |
| Gate 1 — revenue | Modeled levy $9,734,437 = FY2027 certified base × $0.80/$100. Match is **by construction** (scalar calibrated to that base); the independent checks are the pre-calibration gap (+12.73%, inside the plausible phase-in band) and the un-fitted FY2026 levy landing at +2.74% |
| Gate 2 — distribution | Directionally correct: land-heavy up (Vacant +151.7%, Parking +113.0%), building-heavy down (Townhome −25.6%, Large MF −16.3%, Mixed Use −14.1%) |
| Gate 3 — census | 100.0% of parcels matched to Dorchester County 2022 block groups |
| Gate 4 — outputs | 7 of 7 PNGs + `metrics_summary.md` + `parcel_map.html`; `analysis/maps/cambridge_md.parquet` (6,972 parcels, `parcel_id` 100% populated, `parcel_url` intentionally null — see Section 1) |
| Gate 5 — artifact scan | **One artifact found:** condo land is a flat 30% assessor ratio (no per-parcel signal, 17.8% of land base). Everything else clean. Details below. |
| SFR median change (inland) | **−7.0%** |
| SFR median change (waterfront) | **+4.4%** |
| Vacant land median change | **+151.7%** (= the land-only ceiling, correct for bare land) |
| Land millage (4:1) | 20.1353 per $1,000 (= $2.0135/$100) |
| Improvement millage (4:1) | 5.0338 per $1,000 (= $0.5034/$100) |

### Gate 5 — artifact scan, worked

- **No ceiling clustering.** Only `Vacant Land` sits at the +151.7% land-only ceiling, and it is
  100% zero-building by construction. `Transportation - Parking` (+113.0%) sits *below* the ceiling
  with a 20.5% building share — genuinely land-heavy surface lots, not mis-valued buildings.
- **No missing improvement values** (the Seattle failure mode). Of 5,087 taxable parcels with
  structure square footage > 200, only **4** carry an improvement value ≤ $1,000. Improvement
  $/sqft has a plausible distribution (median $85, 5–95% range $22–$158) for this market.
- **The 1,292 zero-improvement parcels are genuinely vacant**, not data gaps: 4 of them have any
  structure sqft, and none has a year built. Cambridge really does carry ~18% vacant lots — a
  shrinking Eastern Shore city with substantial in-town vacancy. This is the finding, not a bug.
- **⚠️ Condominiums (+19.5%) ARE an artifact — a flat assessor land ratio.** Not the St. Paul
  token-land failure (see the next bullet), but the **Maricopa/Phoenix** one. SDAT books condo land
  at a flat **30.0% of total value**: land-share IQR is **0.0002** (p25 0.2997, p75 0.2999) with
  82.9% of units within half a point of exactly 30%, against an IQR of 0.156 for houses and 0.268
  for commercial. The consequence is that the condo outcome has a p10–p90 spread of **1.3
  percentage points** versus **175 points** for houses — it is one number repeated 591 times and
  carries **no per-parcel information**. Because condos are **17.8% of the city land base**, this
  also propagates: Section 5b shows the inland-SFH median moving between **−4.1% and −9.2%**
  (as modelled: −7.0%) and the land millage between 19.66 and 20.77, across plausible alternative
  condo land shares. The
  30% *level* is plausible for waterfront condo product, so SDAT's figures are kept rather than
  replaced by a guess — but the condo row must be read as a statement about an assessment
  convention, not about land economics, and every other category median carries the Section 5b
  error bar. (This is the same conclusion the Phoenix model reached: a split ratio cannot repair a
  collapsed land signal.)
- **No condo collapse (merge-down) is needed, and none was applied.** Verified directly rather
  than assumed: condo unit polygons do **not** stack (sum of areas 9.01 ac = union 9.01 ac, zero
  identical geometries), and **zero** non-condo parcels overlap the condo footprints by more than
  30%, so there is no parent tract carrying the same land a second time. Unlike Ramsey County,
  SDAT gives each unit a real allocated land value (median $80,600, not a $1,000 token), so the
  St. Paul `PlatID` collapse would have nothing to fix here. The defect is the *flatness* of that
  allocation, which collapsing would not repair.
- **Tiny-N caution:** Hotel (3), Other Residential (2) — not interpretable.
- **Agricultural (3) is small only because of preferential assessment, not because Cambridge
  has little farmland.** 18 parcels carry `LU='A'` and 15 of them have no improvement value, so
  the standard `$0 improvement -> Vacant Land` override moves them to Vacant Land, leaving 3 in
  the Agricultural row. Do not read that 3 as "ag is negligible here" — see limitation 6.

### The waterfront split, and an independent validation of it

Applied: **176 of 3,751 single-family parcels (4.7%)** are within 100 m of significant water,
clearing the skill's 2% gate. Waterfront homes go **+4.4%** while inland homes go **−7.0%** — the
signal the split exists to expose, and it would have been invisible in a single blended median.

The computed flag was checked against SDAT's own independent water-influence factor (`PFLW`), and
the two agree closely at both tiers — which is real corroboration, since they share no inputs
(TIGER/Line hydrography vs. an assessor's field judgement):

| Definition | SFH flagged | Median land (flagged vs not) | Ratio |
|---|---|---|---|
| `waterfront_30m` (computed) | 47 (1.3%) | $191,000 vs $30,000 | 6.4× |
| `PFLW = 1` "waterfront" (SDAT) | 45 (1.2%) | $191,000 vs $30,000 | 6.4× |
| `waterfront_100m` (computed, **used**) | 176 (4.7%) | $63,000 vs $26,050 | 2.4× |
| `PFLW ∈ {1,2}` (SDAT) | 123 (3.3%) | $66,000 vs $27,600 | 2.4× |

We keep the canonical `waterfront_100m` for cross-city comparability. The tighter 30 m tier
isolates true frontage (6.4× the inland land value) but falls below the 2% gate.

### Known limitations

1. **Phase-in and homestead are modeled in aggregate, not per parcel.** Maryland taxes phased-in
   assessed value; MD iMAP publishes only the new full cash appraisal (`NFM*`), and per-parcel
   phased-in values sit behind SDAT's bot-protected detail pages. We close the 12.73% gap with one
   uniform scalar (0.8871). Section 4b proves this moves **dollars only** — solved millages are
   bit-identical and every per-parcel percentage change is unchanged — so the distributional
   result does not depend on it. What it cannot capture: Cambridge's 5%/yr homestead cap bites
   harder on long-tenured owner-occupants than on commercial property, so *individual* homeowners'
   current dollar bills are somewhat overstated relative to commercial ones.
2. **City levy only.** Dorchester County ($0.9567/$100 inside Cambridge) and the State
   ($0.1120/$100) are untouched, so a real Cambridge tax bill changes by less in percentage terms
   than this model's figures, which are city-portion-only.
3. **Business personal property ($1.69/$100) and public utility / railroad operating property are
   out of scope** — they are not real property and not in the parcel file. Cambridge's FY2025
   budget booked ~$895k personal property and ~$460k railroad/utility revenue alongside ~$8.2M
   real estate; only the real-property levy is reformed here.
4. **The FY2027 rate ($0.80/$100, Ordinance 1272) is sourced from press coverage**, not yet from a
   primary city document or the SDAT rate table (not published for FY2027 at time of writing). The
   FY2026 rate of $0.8096 *is* primary-sourced. Because the model is revenue-neutral, the rate
   level affects only dollar magnitudes, never the split-rate distribution.
5. **Condominium results carry no per-parcel signal**, and this is the model's largest single
   uncertainty. See the Gate 5 note and Section 5b: SDAT's flat 30% land ratio makes all 591 condos
   move together, and because they are 17.8% of the land base it widens the error bar on every
   other category. If Cambridge wanted a decision-grade number, an independent land-value study of
   the harbor-front condo sites is the highest-value thing to commission.
6. **⚠️ Maryland's agricultural use assessment removes the city's largest block of
   development-zoned land from the LVT signal.** This is the most policy-relevant limitation in the
   model. Md. Tax-Prop. § 8-209 assesses actively farmed land at its *agricultural use value*, not
   market value. In Cambridge that means **710 acres — 19.1% of the 3,710 taxable acres in the
   model — carries just 0.27% of its taxable land value**. Per acre that is a median of **$399** (aggregate
   $1,041) against a median $93,913/acre for residential land and $136,488/acre for commercial: a
   **235x** gap on medians. Those
   parcels are not remote farmland; they are zoned for development inside city limits (B2 197.6 ac,
   R-1 185.5 ac, SR 109.0 ac, R-2 25.9 ac, C-2 22.1 ac). The whole point of a land value tax is to
   make holding development-ready land at low intensity expensive, and the use assessment exempts
   exactly that land from the shift: all 18 parcels together pay about **$10.0k of the $9.73M
   levy** today, and the model's revenue-neutral solver leaves them there. So this model
   **structurally understates** the LVT burden on Cambridge's biggest land-banking opportunity, and
   the understatement is a feature of Maryland law, not of the data. Two consequences worth stating
   plainly: (a) the reported Vacant Land +151.7% is computed on assessed values that, for the ag
   subset, bear no relation to market land value; (b) a real Cambridge LVT would have to decide
   whether to keep § 8-209 inside the city — a policy question this model does not answer and
   cannot, since the market land value of those 710 acres is not in the assessment roll at all.
   Modeling that scenario needs an independent land appraisal, not a reweighting of SDAT figures.
   **And § 8-209 is the sharpest case of a wider pattern, not the whole of it:** assessed land
   $/acre falls ~6x from small lots to 20+ acre tracts *even after excluding ag-assessed parcels*
   (14x before excluding them), and parcels of 5+ acres hold **37% of taxable acreage but 2.4% of
   taxable land value**. Some of
   that is real (unserviced acreage is genuinely worth less per acre than a building lot), but the
   net effect is that this model understates the LVT shift onto large landholders generally — the
   direction of every large-parcel result here is right while its magnitude is conservative.
7. **`Other Commercial` (27) and `Other Residential` (2)** remain as residual buckets at 0.4% of
   taxable parcels, below the skill's 10% refinement threshold.